# 1. Import libraries

In [ ]:
# Standard library imports
import sys
from pathlib import Path
from dataclasses import dataclass

# Data manipulation libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import altair as alt

# Statistics and ML preprocessing
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder


# Custom module imports
sys.path.append(str(Path.cwd().parent / "src"))
from plot import create_scatter_correlation_plot

# Set up plotting parameters
plt.style.use("../../config/DIT_HAP.mplstyle")
COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]
AX_WIDTH, AX_HEIGHT = plt.rcParams["figure.figsize"]

# 2. Configurations

In [ ]:
@dataclass
class config:
    output_dir: Path = Path("../../results/HD_DIT_HAP_generationRAW/22_machine_learning_modeling")

    feature_value_file: Path = Path("../../resources/pombe_features/pombe_coding_gene_protein_features.tsv")
    target_value_file: Path = Path("../../results/HD_DIT_HAP_generationRAW/18_gene_level_clustering/kmeans_cluster_result.tsv")

    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.feature_values: pd.DataFrame = pd.read_csv(self.feature_value_file, sep="\t", index_col=0)
        self.target_values: pd.DataFrame = pd.read_csv(self.target_value_file, sep="\t", index_col=0)[["um", "lam", "revised_cluster"]]
        self.feature_with_target: pd.DataFrame = pd.concat([self.feature_values, self.target_values], axis=1)

cfg = config()

# 3. Data Info

In [ ]:
datasets = {
    "all": {
        "features": cfg.feature_values,
        "target": cfg.target_values,
        "feature_with_target": cfg.feature_with_target,
    },
    "DR_gt_0.3": {
        "features": cfg.feature_with_target[cfg.feature_with_target["um"] > 0.3][cfg.feature_values.columns],
        "target": cfg.feature_with_target[cfg.feature_with_target["um"] > 0.3][cfg.target_values.columns],
        "feature_with_target": cfg.feature_with_target[cfg.feature_with_target["um"] > 0.3],
    },
    "DR_le_0.3": {
        "features": cfg.feature_with_target[cfg.feature_with_target["um"] <= 0.3][cfg.feature_values.columns],
        "target": cfg.feature_with_target[cfg.feature_with_target["um"] <= 0.3][cfg.target_values.columns],
        "feature_with_target": cfg.feature_with_target[cfg.feature_with_target["um"] <= 0.3],
    }
}

for name, dataset in datasets.items():
    print("*** Dataset:", name)
    print("Shape:", dataset["feature_with_target"].shape)
    print(dataset["features"].info())
    print(dataset["target"].info())

# 4. Feature Distribution and Transformations

In [ ]:
def data_transformation(values: pd.Series, transformation: str = "No processing") -> pd.DataFrame:
    """Apply specified transformation to the data series."""

    match transformation:
        case 'StandardScaler':
            transformed = StandardScaler().fit_transform(values.values.reshape(-1, 1))
            prefix = 'std_scaled_'
            transformed = pd.DataFrame(transformed, index=values.index, columns=[values.name])
        case 'OneHotEncoder':
            transformed = pd.get_dummies(values, prefix=values.name, dummy_na=True).astype(int)
            prefix = 'one_hot_'
        case 'np.log1p':
            transformed = np.log1p(values)
            transformed = StandardScaler().fit_transform(transformed.values.reshape(-1, 1))
            prefix = 'log1p_std_scaled_'
            transformed = pd.DataFrame(transformed, index=values.index, columns=[values.name])
        case 'No processing':
            transformed = values.to_frame()
            prefix = ''
        case _:
            raise ValueError(f"Unsupported transformation: {transformation}")
    transformed.columns = [f"{prefix}{col}" for col in transformed.columns] if isinstance(transformed, pd.DataFrame) else [f"{prefix}{values.name}"]
    return transformed

## 4.1 Numerical Features Distribution

In [ ]:
numerical_features = cfg.feature_values.select_dtypes(include=[np.number]).columns.tolist()
total_feature_plots = len(numerical_features)
n_cols = 4
n_rows = (total_feature_plots + n_cols - 1) // n_cols

with PdfPages(cfg.output_dir / "numerical_feature_distributions.pdf") as pdf:
    for name, dataset in datasets.items():
        dataset["features"][numerical_features].hist(bins=50, figsize=(n_cols*AX_WIDTH, n_rows*AX_HEIGHT), layout=(n_rows, n_cols))
        plt.suptitle(f"Numerical Feature Distributions - {name}", y=1.02)
        plt.tight_layout()
        pdf.savefig()
        plt.close()

In [ ]:
selected_features_and_transformations = {
    # DNA information
    'Relative_distance_from_telomere': 'StandardScaler',
    'Relative_distance_from_centromere': 'StandardScaler',
    'Gene_length': 'np.log1p',
    'GC_content_of_gene': 'StandardScaler',
    'CDS_number': 'np.log1p',
    'GC_content_of_CDS': 'StandardScaler',
    'Fraction_of_CDS': 'No processing',
    'GC3': 'StandardScaler',
    'Intron_number': 'np.log1p',
    'GC_content_of_intron': 'No processing',
    'Total_intron_length': 'np.log1p',
    'Average_intron_length': 'np.log1p',
    'Length_of_first_intron': 'np.log1p',
    'GC_contents_of_first_intron': 'No processing',
    'ENC': 'StandardScaler',

    # RNA information
    'mean_EMM_Nitrogen_Starved_Cell_RNA_Abundance': 'np.log1p',
    'mean_EMM_Proliferating_Cell_RNA_Abundance': 'np.log1p',
    'cv_EMM_Nitrogen_Starved_Cell_RNA_Abundance': 'np.log1p',
    'cv_EMM_Proliferating_Cell_RNA_Abundance': 'np.log1p',
    'tAIg': 'StandardScaler',
    'mRNA_half_life_minutes': 'np.log1p',
    'mRNA_synthesis_rate_per_minute': 'np.log1p',
    
    # Protein information
    'Mass (kDa)': 'np.log1p',
    'pI': 'StandardScaler',
    'Charge': 'StandardScaler',
    'Residues': 'np.log1p',
    'CAI': 'StandardScaler',
    'aromaticity': 'StandardScaler',
    'aliphatic_index': 'StandardScaler',
    'gravy': 'StandardScaler',
    'flexibility': 'StandardScaler',
    'instability_index': 'StandardScaler',
    'aa_percent_Ala': 'StandardScaler',
    'aa_percent_Cys': 'StandardScaler',
    'aa_percent_Asp': 'StandardScaler',
    'aa_percent_Glu': 'StandardScaler',
    'aa_percent_Phe': 'StandardScaler',
    'aa_percent_Gly': 'StandardScaler',
    'aa_percent_His': 'StandardScaler',
    'aa_percent_Ile': 'StandardScaler',
    'aa_percent_Lys': 'StandardScaler',
    'aa_percent_Leu': 'StandardScaler',
    'aa_percent_Met': 'StandardScaler',
    'aa_percent_Asn': 'StandardScaler',
    'aa_percent_Pro': 'StandardScaler',
    'aa_percent_Gln': 'StandardScaler',
    'aa_percent_Arg': 'StandardScaler',
    'aa_percent_Ser': 'StandardScaler',
    'aa_percent_Thr': 'StandardScaler',
    'aa_percent_Val': 'StandardScaler',
    'aa_percent_Trp': 'StandardScaler',
    'aa_percent_Tyr': 'StandardScaler',
    # 'copies_per_cell_EMM_Proliferating_Cell': 'np.log1p',
    # 'copies_per_cell_EMMN_Quiescent_Cell': 'np.log1p',
    # 't1/2 (min)': 'StandardScaler',
    'mean_pLDDT': 'StandardScaler',
    'cv_pLDDT': 'StandardScaler',
    'PFAM_domain_count': 'np.log1p',

    # Evolutionary information
    'japonicus_ortholog_count': 'np.log1p',
    'cerevisiae_ortholog_count': 'np.log1p',
    'human_ortholog_count': 'np.log1p',
    'paralog_count': 'np.log1p',
    'evolutionary_rate': 'StandardScaler',
    'mean.phylop': 'StandardScaler',
    'diversity.S': 'StandardScaler',
    'diversity.Pi': 'StandardScaler',
    'diversity.Theta': 'StandardScaler',
    'diversity.Tajima_D': 'StandardScaler',

    # Network information
    'GO_term_richness': 'np.log1p',
    'PPI_degree': 'np.log1p',
    'GI_degree': 'np.log1p'
}

In [ ]:
selected_transformed_numerical_features = {}

for name, dataset in datasets.items():
    selected_transformed_numerical_features[name] = pd.DataFrame()
    for feature, transformation in selected_features_and_transformations.items():
        transformed = data_transformation(dataset["features"][feature], transformation)
        selected_transformed_numerical_features[name] = pd.concat([selected_transformed_numerical_features[name], transformed], axis=1)

In [ ]:
total_feature_plots = len(selected_features_and_transformations)
n_cols = 4
n_rows = (total_feature_plots + n_cols - 1) // n_cols

with PdfPages(cfg.output_dir / "selected_transformed_numerical_feature_distributions.pdf") as pdf:
    for name, dataset in datasets.items():
        selected_transformed_numerical_features[name].hist(bins=50, figsize=(n_cols*AX_WIDTH, n_rows*AX_HEIGHT), layout=(n_rows, n_cols))
        plt.suptitle(f"Selected Transformed Numerical Feature Distributions - {name}", y=1.02)
        plt.tight_layout()
        pdf.savefig()
        plt.close()

## 4.2 Categorical Features Distribution

In [ ]:
categorical_features = cfg.feature_values.select_dtypes(include=['object', 'category']).columns.tolist()
n_cols = 5
n_rows = 1


with PdfPages(cfg.output_dir / "categorical_feature_distributions.pdf") as pdf:
    for name, dataset in datasets.items():
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*AX_WIDTH, n_rows*AX_HEIGHT))
        for ax, col in zip(axes, ['Chromosome','Strand','FYPOviability','Gene dispensability. This study','Category']):
            dataset["features"][col].value_counts().plot(kind='bar', ax=ax)
            ax.set_title(col)
        
        plt.suptitle(f"Categorical Feature Distributions - {name}", y=1.02)
        plt.tight_layout()
        pdf.savefig()
        plt.close()

In [ ]:
selected_categorical_features = ['Chromosome','Strand']
selected_transformed_categorical_features = {}

for name, dataset in datasets.items():
    selected_transformed_categorical_features[name] = pd.DataFrame()
    for feature in selected_categorical_features:
        transformed = data_transformation(dataset["features"][feature], 'OneHotEncoder')
        selected_transformed_categorical_features[name] = pd.concat([selected_transformed_categorical_features[name], transformed], axis=1)

## 4.3 Target value Distribution

In [ ]:
with PdfPages(cfg.output_dir / "selected_target_distributions.pdf") as pdf:
    for name, dataset in datasets.items():
        fig, axes = plt.subplots(1, 3, figsize=(3*AX_WIDTH, AX_HEIGHT))

        dataset["target"][["um", "lam"]].hist(bins=50, ax=axes[:2])

        dataset["target"]["revised_cluster"].value_counts().plot(kind='bar', ax=axes[2])
        
        plt.suptitle(f"Target Value Distributions - {name}", y=1.02)
        plt.tight_layout()
        pdf.savefig()
        plt.close()

# 5. Concat transformed data

In [ ]:
concated_transformed_data = {}

for name in datasets.keys():
    concated_transformed_features = pd.concat(
        [
            selected_transformed_numerical_features[name], 
            selected_transformed_categorical_features[name]
        ],
        axis=1
    ).rename_axis('gene_systematic_id')
    concated_transformed_features.T.to_csv(cfg.output_dir / f"concated_transformed_features_{name}_transposed.csv", index=True)

    datasets[name]["target"].rename_axis('gene_systematic_id').to_csv(cfg.output_dir / f"target_values_{name}.csv", index=True)

    concated_transformed_data[name] = pd.concat(
        [concated_transformed_features, datasets[name]["target"]],
        axis=1
    ).rename_axis('gene_systematic_id')
    concated_transformed_data[name].to_csv(cfg.output_dir / f"concated_transformed_data_{name}.csv", index=True)

    noNA_transformed_data = concated_transformed_data[name].dropna()
    noNA_transformed_data[concated_transformed_features.columns].T.to_csv(cfg.output_dir / f"concated_transformed_features_{name}_transposed_noNA.csv", index=True)
    noNA_transformed_data[datasets[name]["target"].columns].rename_axis('gene_systematic_id').to_csv(cfg.output_dir / f"target_values_{name}_noNA.csv", index=True)

# 6. Plot - feature vs target

In [ ]:
for target in ["um", "lam"]:
    for name, dataset in concated_transformed_data.items():
        output_path = cfg.output_dir / f"feature_vs_{target}_{name}_correlation_plots.pdf"
        with PdfPages(output_path) as pdf:
            columns = selected_transformed_numerical_features[name].columns.tolist()
            n_cols = 4
            n_rows = (len(columns) + n_cols - 1) // n_cols

            fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*AX_WIDTH, n_rows*AX_HEIGHT))
            axes = axes.flatten()

            for ax, feature in zip(axes, columns):
                create_scatter_correlation_plot(
                    dataset[feature],
                    dataset[target],
                    ax,
                    show_diagonal=False
                )
                ax.set_xlabel(feature)
                ax.set_ylabel(target)

                ax.set_rasterized(True)
            
            plt.suptitle(f"Feature vs {target} Correlation Plots - {name}", y=1.02)
            plt.tight_layout()
            pdf.savefig()
            plt.close()
